In [29]:
# =============================================================================
# CONFUSION MATRIX EVALUATION — Kaggle Notebook Version
# ResNet50 + DenseNet121 + Faster-RCNN  |  TN5000  |  Pascal VOC Format
# =============================================================================

import os
import xml.etree.ElementTree as ET

import cv2
import numpy as np
import torch
import torchvision.transforms as T
import torchvision.models as models
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from PIL import Image
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.transforms import functional as TF
from torchvision.ops import box_iou
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_auc_score
)


In [30]:
# =============================================================================
# 1. KAGGLE PATHS  — update dataset names to match what you uploaded
# =============================================================================

# Weights — update 'your-weights-dataset' to your Kaggle dataset name
WEIGHTS_ROOT = "/kaggle/input/models/mok18976/thyroid/pytorch/default/1"

RCNN_WEIGHTS_PATH      = os.path.join(WEIGHTS_ROOT, "best_fasterrcnn.pth")
DENSENET_WEIGHTS_PATHS = [
    os.path.join(WEIGHTS_ROOT, "best_densenet_fold1.pth"),
    os.path.join(WEIGHTS_ROOT, "best_densenet_fold2.pth"),
    os.path.join(WEIGHTS_ROOT, "best_densenet_fold3.pth"),
]
RESNET_WEIGHTS_PATHS = [
    os.path.join(WEIGHTS_ROOT, "best_resnet50_fold1.pth"),
    os.path.join(WEIGHTS_ROOT, "best_resnet50_fold2.pth"),
    os.path.join(WEIGHTS_ROOT, "best_resnet50_fold3.pth"),
]

# Dataset — update 'your-tn5000-dataset' to your Kaggle dataset name
DATASET_ROOT    = "/kaggle/input/datasets/abdullahelafifi/main-data/Main data"
ANNOTATIONS_DIR = os.path.join(DATASET_ROOT, "Annotations")
IMAGES_DIR      = os.path.join(DATASET_ROOT, "JPEGImages")
TEST_LIST_FILE  = os.path.join(DATASET_ROOT, "ImageSets", "Main", "test.txt")

In [31]:
# =============================================================================
# 2. VERIFY PATHS  — run this first to confirm everything is found
# =============================================================================

print("=== PATH CHECK ===")
print(f"RCNN weights     : {'OK' if os.path.exists(RCNN_WEIGHTS_PATH) else 'NOT FOUND'} | {RCNN_WEIGHTS_PATH}")
for p in DENSENET_WEIGHTS_PATHS:
    print(f"DenseNet weights : {'OK' if os.path.exists(p) else 'NOT FOUND'} | {p}")
for p in RESNET_WEIGHTS_PATHS:
    print(f"ResNet weights   : {'OK' if os.path.exists(p) else 'NOT FOUND'} | {p}")
print(f"Annotations dir  : {'OK' if os.path.exists(ANNOTATIONS_DIR) else 'NOT FOUND'} | {ANNOTATIONS_DIR}")
print(f"Images dir       : {'OK' if os.path.exists(IMAGES_DIR) else 'NOT FOUND'} | {IMAGES_DIR}")
print(f"test.txt         : {'OK' if os.path.exists(TEST_LIST_FILE) else 'NOT FOUND'} | {TEST_LIST_FILE}")
print("==================")

=== PATH CHECK ===
RCNN weights     : OK | /kaggle/input/models/mok18976/thyroid/pytorch/default/1/best_fasterrcnn.pth
DenseNet weights : OK | /kaggle/input/models/mok18976/thyroid/pytorch/default/1/best_densenet_fold1.pth
DenseNet weights : OK | /kaggle/input/models/mok18976/thyroid/pytorch/default/1/best_densenet_fold2.pth
DenseNet weights : OK | /kaggle/input/models/mok18976/thyroid/pytorch/default/1/best_densenet_fold3.pth
ResNet weights   : OK | /kaggle/input/models/mok18976/thyroid/pytorch/default/1/best_resnet50_fold1.pth
ResNet weights   : OK | /kaggle/input/models/mok18976/thyroid/pytorch/default/1/best_resnet50_fold2.pth
ResNet weights   : OK | /kaggle/input/models/mok18976/thyroid/pytorch/default/1/best_resnet50_fold3.pth
Annotations dir  : OK | /kaggle/input/datasets/abdullahelafifi/main-data/Main data/Annotations
Images dir       : OK | /kaggle/input/datasets/abdullahelafifi/main-data/Main data/JPEGImages
test.txt         : OK | /kaggle/input/datasets/abdullahelafifi/main-

In [32]:
# =============================================================================
# 3. CONSTANTS (identical to training)
# =============================================================================

DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLAHE_CLIP        = 2.0
CLAHE_GRID        = (8, 8)
CROP_MARGIN       = 30
RESIZE_SIZE       = 224
RCNN_SCORE_THRESH = 0.4
NMS_IOU_THRESH    = 0.3
MALIGNANT_THRESH  = 0.20
RCNN_LABEL_NAMES  = {1: "benign", 2: "malignant"}
LABEL_MAP         = {"0": 0, "1": 1}

print(f"Running on: {DEVICE}")

Running on: cuda


In [33]:
# =============================================================================
# 4. PREPROCESSING
# =============================================================================

def apply_clahe(img_bgr):
    lab     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe   = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)
    l_clahe = clahe.apply(l)
    img_clahe = cv2.cvtColor(cv2.merge([l_clahe, a, b]), cv2.COLOR_LAB2BGR)
    blurred   = cv2.GaussianBlur(img_clahe, (0, 0), sigmaX=2)
    sharpened = cv2.addWeighted(img_clahe, 1.3, blurred, -0.3, 0)
    return sharpened

def load_and_preprocess(image_path):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot open: {image_path}")
    img_bgr = apply_clahe(img_bgr)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(img_rgb)

In [34]:
# =============================================================================
# 5. MODEL BUILDERS
# =============================================================================

def build_rcnn(num_classes=3):
    return fasterrcnn_resnet50_fpn_v2(weights=None, num_classes=num_classes)

def build_densenet():
    model = models.densenet121(weights=None)
    num_features     = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(num_features, 2)
    )
    return model

def build_resnet50():
    model = models.resnet50(weights=None)
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(512, 2)
    )
    return model

In [35]:
# =============================================================================
# 6. LOAD MODELS
# =============================================================================

def load_models():
    print("Loading RCNN...")
    rcnn = build_rcnn()
    rcnn.load_state_dict(torch.load(RCNN_WEIGHTS_PATH, map_location=DEVICE))
    rcnn.to(DEVICE).eval()

    print("Loading DenseNet ensemble...")
    dn_models = []
    for i, path in enumerate(DENSENET_WEIGHTS_PATHS):
        m = build_densenet()
        m.load_state_dict(torch.load(path, map_location=DEVICE))
        m.to(DEVICE).eval()
        dn_models.append(m)
        print(f"  DenseNet fold {i+1} loaded")

    print("Loading ResNet50 ensemble...")
    rn_models = []
    for i, path in enumerate(RESNET_WEIGHTS_PATHS):
        m = build_resnet50()
        m.load_state_dict(torch.load(path, map_location=DEVICE))
        m.to(DEVICE).eval()
        rn_models.append(m)
        print(f"  ResNet50 fold {i+1} loaded")

    print("All models ready.\n")
    return rcnn, dn_models, rn_models

In [36]:
# =============================================================================
# 7. TRANSFORMS (identical to training)
# =============================================================================

val_transform = T.Compose([
    T.Resize((RESIZE_SIZE, RESIZE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

tta_transform = T.Compose([
    T.Resize((RESIZE_SIZE, RESIZE_SIZE)),
    T.FiveCrop(int(RESIZE_SIZE * 0.9)),
    T.Lambda(lambda crops: torch.stack([
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])(T.ToTensor()(c))
        for c in crops
    ]))
])

In [37]:
# =============================================================================
# 8. INFERENCE (silent — no plots, no prints)
# =============================================================================

def detect_nodules(rcnn_model, img_pil):
    tensor = TF.to_tensor(img_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        preds = rcnn_model(tensor)[0]
    boxes  = preds["boxes"].cpu()
    labels = preds["labels"].cpu()
    scores = preds["scores"].cpu()
    keep   = scores >= RCNN_SCORE_THRESH
    return boxes[keep], labels[keep], scores[keep]

def apply_nodule_nms(boxes, labels, scores):
    if len(boxes) == 0:
        return []
    order  = scores.argsort(descending=True)
    boxes  = boxes[order]
    labels = labels[order]
    scores = scores[order]
    kept   = []
    active = [True] * len(boxes)
    for i in range(len(boxes)):
        if not active[i]:
            continue
        kept.append((boxes[i], labels[i], scores[i]))
        for j in range(i + 1, len(boxes)):
            if not active[j]:
                continue
            if box_iou(boxes[i].unsqueeze(0), boxes[j].unsqueeze(0)).item() >= NMS_IOU_THRESH:
                active[j] = False
    return kept

def crop_nodule(img_pil, box):
    W, H = img_pil.size
    x1, y1, x2, y2 = map(int, box.tolist())
    x1 = max(0, x1 - CROP_MARGIN)
    y1 = max(0, y1 - CROP_MARGIN)
    x2 = min(W, x2 + CROP_MARGIN)
    y2 = min(H, y2 + CROP_MARGIN)
    return img_pil.crop((x1, y1, x2, y2))

def classify_crop(crop_pil, dn_models, rn_models):
    crop_rgb   = crop_pil.convert("RGB")
    probs_list = []

    tensor_dn = val_transform(crop_rgb).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        for m in dn_models:
            out   = m(tensor_dn)
            probs = torch.softmax(out, dim=1)[0].cpu().numpy()
            probs_list.append(probs)

    tensor_rn = tta_transform(crop_rgb).to(DEVICE)
    with torch.no_grad():
        for m in rn_models:
            out      = m(tensor_rn)
            probs_5  = torch.softmax(out, dim=1)
            avg_prob = probs_5.mean(dim=0).cpu().numpy()
            probs_list.append(avg_prob)

    avg_probs = np.mean(probs_list, axis=0)
    mal_prob  = float(avg_probs[1])
    cnn_label = "malignant" if mal_prob >= MALIGNANT_THRESH else "benign"
    return cnn_label, mal_prob

def fuse_predictions(cnn_label, mal_prob, rcnn_label_int):
    rcnn_label = RCNN_LABEL_NAMES.get(rcnn_label_int, "benign")
    if cnn_label == rcnn_label:
        return cnn_label, mal_prob
    if 0.35 <= mal_prob <= 0.50:
        return rcnn_label, mal_prob
    return cnn_label, mal_prob

def predict_image(img_pil, rcnn_model, dn_models, rn_models):
    boxes, labels, scores = detect_nodules(rcnn_model, img_pil)
    if len(boxes) == 0:
        return 0, 0.0

    kept       = apply_nodule_nms(boxes, labels, scores)
    best_label = "benign"
    best_prob  = 0.0

    for (box, rcnn_label_t, _) in kept:
        crop_pil              = crop_nodule(img_pil, box)
        cnn_label, mal_prob   = classify_crop(crop_pil, dn_models, rn_models)
        final_label, mal_prob = fuse_predictions(cnn_label, mal_prob, rcnn_label_t.item())
        if final_label == "malignant" or mal_prob > best_prob:
            best_label = final_label
            best_prob  = mal_prob

    return (1 if best_label == "malignant" else 0), best_prob

In [38]:
# =============================================================================
# 9. GROUND TRUTH FROM VOC XML
# =============================================================================

def get_gt_label_from_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        if LABEL_MAP.get(name, 0) == 1:
            return 1
    return 0

In [39]:
# =============================================================================
# 10. LOAD TEST SET
# =============================================================================

with open(TEST_LIST_FILE, "r") as f:
    test_ids = [line.strip() for line in f if line.strip()]

image_paths  = []
ground_truth = []

for img_id in test_ids:
    xml_path = os.path.join(ANNOTATIONS_DIR, img_id + ".xml")
    img_path = os.path.join(IMAGES_DIR,      img_id + ".jpg")
    if not os.path.exists(img_path):
        img_path = os.path.join(IMAGES_DIR,  img_id + ".png")
    if not os.path.exists(xml_path) or not os.path.exists(img_path):
        continue
    image_paths.append(img_path)
    ground_truth.append(get_gt_label_from_xml(xml_path))

print(f"Test set : {len(image_paths)} images")
print(f"Benign   : {ground_truth.count(0)}")
print(f"Malignant: {ground_truth.count(1)}\n")

Test set : 1000 images
Benign   : 269
Malignant: 731



In [40]:
# =============================================================================
# 11. RUN EVALUATION
# =============================================================================

rcnn_model, dn_models, rn_models = load_models()

predictions = []
mal_probs   = []
no_detect   = 0
errors      = 0

for i, img_path in enumerate(image_paths):
    try:
        img_pil    = load_and_preprocess(img_path)
        pred, prob = predict_image(img_pil, rcnn_model, dn_models, rn_models)
        if pred == 0 and prob == 0.0:
            no_detect += 1
    except Exception as e:
        pred, prob = 0, 0.0
        errors += 1

    predictions.append(pred)
    mal_probs.append(prob)

    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(image_paths)} processed...")

print(f"\nDone. No-detection fallback: {no_detect}  |  Errors: {errors}")

Loading RCNN...
Loading DenseNet ensemble...
  DenseNet fold 1 loaded
  DenseNet fold 2 loaded
  DenseNet fold 3 loaded
Loading ResNet50 ensemble...
  ResNet50 fold 1 loaded
  ResNet50 fold 2 loaded
  ResNet50 fold 3 loaded
All models ready.

  20/1000 processed...
  40/1000 processed...
  60/1000 processed...
  80/1000 processed...
  100/1000 processed...
  120/1000 processed...
  140/1000 processed...
  160/1000 processed...
  180/1000 processed...
  200/1000 processed...
  220/1000 processed...
  240/1000 processed...
  260/1000 processed...
  280/1000 processed...
  300/1000 processed...
  320/1000 processed...
  340/1000 processed...
  360/1000 processed...
  380/1000 processed...
  400/1000 processed...
  420/1000 processed...
  440/1000 processed...
  460/1000 processed...
  480/1000 processed...
  500/1000 processed...
  520/1000 processed...
  540/1000 processed...
  560/1000 processed...
  580/1000 processed...
  600/1000 processed...
  620/1000 processed...
  640/1000 proces

In [41]:
# =============================================================================
# 12. CONFUSION MATRIX
# =============================================================================

cm = confusion_matrix(ground_truth, predictions, labels=[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay(cm, display_labels=["Benign", "Malignant"]).plot(
    ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix (Counts)", fontsize=13)

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
ConfusionMatrixDisplay(cm_norm, display_labels=["Benign", "Malignant"]).plot(
    ax=axes[1], colorbar=False, cmap="Blues", values_format=".1f")
axes[1].set_title("Confusion Matrix (% per True Class)", fontsize=13)

plt.suptitle("ResNet50 + DenseNet121 + Faster-RCNN  |  TN5000 Test Set",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_tn5000.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → /kaggle/working/confusion_matrix_tn5000.png")

Saved → /kaggle/working/confusion_matrix_tn5000.png


In [42]:
# =============================================================================
# 13. METRICS
# =============================================================================

TN, FP, FN, TP = cm.ravel()

sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
precision   = TP / (TP + FP) if (TP + FP) > 0 else 0
npv         = TN / (TN + FN) if (TN + FN) > 0 else 0
accuracy    = (TP + TN) / len(ground_truth)
f1          = (2 * precision * sensitivity / (precision + sensitivity)
               if (precision + sensitivity) > 0 else 0)
try:
    auc     = roc_auc_score(ground_truth, mal_probs)
    auc_str = f"{auc:.4f}"
except Exception:
    auc_str = "N/A"

print("\n" + "="*52)
print("   EVALUATION RESULTS — TN5000 TEST SET")
print("="*52)
print(f"  Total images           : {len(ground_truth)}")
print(f"  TP (Malignant correct) : {TP}")
print(f"  TN (Benign correct)    : {TN}")
print(f"  FP (Benign→Malignant)  : {FP}")
print(f"  FN (Malignant missed)  : {FN}")
print(f"  {'─'*40}")
print(f"  Accuracy               : {accuracy:.4f}")
print(f"  Sensitivity (Recall)   : {sensitivity:.4f}")
print(f"  Specificity            : {specificity:.4f}")
print(f"  Precision (PPV)        : {precision:.4f}")
print(f"  NPV                    : {npv:.4f}")
print(f"  F1-Score               : {f1:.4f}")
print(f"  ROC-AUC                : {auc_str}")
print("="*52)
print(classification_report(ground_truth, predictions,
                             target_names=["Benign", "Malignant"]))


   EVALUATION RESULTS — TN5000 TEST SET
  Total images           : 1000
  TP (Malignant correct) : 649
  TN (Benign correct)    : 177
  FP (Benign→Malignant)  : 92
  FN (Malignant missed)  : 82
  ────────────────────────────────────────
  Accuracy               : 0.8260
  Sensitivity (Recall)   : 0.8878
  Specificity            : 0.6580
  Precision (PPV)        : 0.8758
  NPV                    : 0.6834
  F1-Score               : 0.8818
  ROC-AUC                : 0.8545
              precision    recall  f1-score   support

      Benign       0.68      0.66      0.67       269
   Malignant       0.88      0.89      0.88       731

    accuracy                           0.83      1000
   macro avg       0.78      0.77      0.78      1000
weighted avg       0.82      0.83      0.82      1000



In [43]:
# =============================================================================
# THRESHOLD OPTIMIZATION
# =============================================================================
import numpy as np

# First collect raw mal_probs from the combined pipeline run
# (mal_probs list already exists from the main evaluation)

thresholds = np.arange(0.10, 0.60, 0.02)

results_table = []

for thresh in thresholds:
    preds_t = [1 if p >= thresh else 0 for p in mal_probs]
    cm_t    = confusion_matrix(ground_truth, preds_t, labels=[0, 1])
    TN_t, FP_t, FN_t, TP_t = cm_t.ravel()

    sensitivity_t = TP_t / (TP_t + FN_t) if (TP_t + FN_t) > 0 else 0
    specificity_t = TN_t / (TN_t + FP_t) if (TN_t + FP_t) > 0 else 0
    precision_t   = TP_t / (TP_t + FP_t) if (TP_t + FP_t) > 0 else 0
    f1_t          = (2 * precision_t * sensitivity_t /
                    (precision_t + sensitivity_t)
                    if (precision_t + sensitivity_t) > 0 else 0)
    accuracy_t    = (TP_t + TN_t) / len(ground_truth)

    results_table.append({
        "threshold"  : round(float(thresh), 2),
        "TP"         : int(TP_t),
        "TN"         : int(TN_t),
        "FP"         : int(FP_t),
        "FN"         : int(FN_t),
        "Sensitivity": round(sensitivity_t, 4),
        "Specificity": round(specificity_t, 4),
        "Precision"  : round(precision_t,   4),
        "F1"         : round(f1_t,           4),
        "Accuracy"   : round(accuracy_t,     4),
    })

# ── PRINT TABLE ───────────────────────────────────────────────────
print(f"{'Thresh':>7} {'TP':>5} {'TN':>5} {'FP':>5} {'FN':>5} "
      f"{'Sens':>7} {'Spec':>7} {'Prec':>7} {'F1':>7} {'Acc':>7}")
print("─" * 75)
for r in results_table:
    print(f"{r['threshold']:>7.2f} {r['TP']:>5} {r['TN']:>5} "
          f"{r['FP']:>5} {r['FN']:>5} "
          f"{r['Sensitivity']:>7.4f} {r['Specificity']:>7.4f} "
          f"{r['Precision']:>7.4f} {r['F1']:>7.4f} {r['Accuracy']:>7.4f}")

# ── PLOT SENSITIVITY vs SPECIFICITY vs F1 ────────────────────────
thresh_vals   = [r["threshold"]   for r in results_table]
sensitivity_v = [r["Sensitivity"] for r in results_table]
specificity_v = [r["Specificity"] for r in results_table]
f1_v          = [r["F1"]          for r in results_table]
fn_v          = [r["FN"]          for r in results_table]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left — metrics vs threshold
axes[0].plot(thresh_vals, sensitivity_v, "r-o", markersize=4, label="Sensitivity")
axes[0].plot(thresh_vals, specificity_v, "b-o", markersize=4, label="Specificity")
axes[0].plot(thresh_vals, f1_v,          "g-o", markersize=4, label="F1-Score")
axes[0].axvline(x=0.32, color="gray", linestyle="--", label="Current (0.32)")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Score")
axes[0].set_title("Metrics vs Threshold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right — FN count vs threshold (missed malignant)
axes[1].plot(thresh_vals, fn_v, "r-o", markersize=4, label="FN (Missed Malignant)")
axes[1].axvline(x=0.32, color="gray", linestyle="--", label="Current (0.32)")
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("FN Count")
axes[1].set_title("Missed Malignant (FN) vs Threshold")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Threshold Optimization — Combined Pipeline", fontsize=13)
plt.tight_layout()
plt.savefig("/kaggle/working/threshold_optimization.png", dpi=150, bbox_inches="tight")
plt.show()

# ── FIND BEST THRESHOLD FOR EACH GOAL ────────────────────────────
best_f1   = max(results_table, key=lambda r: r["F1"])
best_sens = max(results_table, key=lambda r: r["Sensitivity"])
best_bal  = max(results_table, key=lambda r: r["Sensitivity"] + r["Specificity"])

print("\n" + "="*52)
print("  BEST THRESHOLD RECOMMENDATIONS")
print("="*52)
print(f"  Best F1-Score      : threshold={best_f1['threshold']}  "
      f"F1={best_f1['F1']}  FN={best_f1['FN']}")
print(f"  Best Sensitivity   : threshold={best_sens['threshold']}  "
      f"Sens={best_sens['Sensitivity']}  FN={best_sens['FN']}")
print(f"  Best Balanced      : threshold={best_bal['threshold']}  "
      f"Sens={best_bal['Sensitivity']}  Spec={best_bal['Specificity']}")
print("="*52)

 Thresh    TP    TN    FP    FN    Sens    Spec    Prec      F1     Acc
───────────────────────────────────────────────────────────────────────────
   0.10   687   122   147    44  0.9398  0.4535  0.8237  0.8780  0.8090
   0.12   684   132   137    47  0.9357  0.4907  0.8331  0.8814  0.8160
   0.14   683   142   127    48  0.9343  0.5279  0.8432  0.8864  0.8250
   0.16   679   149   120    52  0.9289  0.5539  0.8498  0.8876  0.8280
   0.18   678   157   112    53  0.9275  0.5836  0.8582  0.8915  0.8350
   0.20   677   159   110    54  0.9261  0.5911  0.8602  0.8920  0.8360
   0.22   673   168   101    58  0.9207  0.6245  0.8695  0.8944  0.8410
   0.24   670   176    93    61  0.9166  0.6543  0.8781  0.8969  0.8460
   0.26   668   178    91    63  0.9138  0.6617  0.8801  0.8966  0.8460
   0.28   662   182    87    69  0.9056  0.6766  0.8838  0.8946  0.8440
   0.30   657   189    80    74  0.8988  0.7026  0.8915  0.8951  0.8460
   0.32   650   192    77    81  0.8892  0.7138  0.8941  0.8